# Setup

This notebook converts strongly labelled data into the info tables required for training and testing Voxaboxen models.

In [1]:
import json
import pandas as pd
import os
from urllib.request import urlretrieve
from pydub import AudioSegment
from tqdm import tqdm

# Step 1 - Create Selection Tables

The selection tables are .txt files with only the following columns required: ``Begin Time (s)``, ``End Time (s)``, ``Annotation``.
They can be created through one of the following options:
* Option A: Parsing a single JSON file into multiple Raven selection tables
* Option B: Formatting existing Raven selection tables

Only run the cells under either Option A or B, before moving onto Step 2.

##  Option A: Parse JSON File

Follow these steps if strongly labelled data has been exported from Soundbox as a JSON file.

In [14]:
JSON_PATH = '/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/annotation-project-17d6a079-0e06-4f72-96fb-6fab723ad827.json'

with open(JSON_PATH, 'r') as file:
    data = json.load(file)

In [15]:
# Map recording uuid to capture ID
recordings = data['recordings']
recordings_df = pd.DataFrame.from_dict(recordings)
uuid_to_capture_map = recordings_df.set_index('uuid')['path'].to_dict()
uuid_to_capture_map = {k: v.split('.')[0] for k,v in uuid_to_capture_map.items()}

In [16]:
# Extract start/end times from sound events
sound_events = data['sound_events']
sound_events_df = pd.DataFrame.from_dict(sound_events)
try:
    sound_events_df["begin_time"] = sound_events_df["geometry"].apply(lambda x: x["coordinates"][0])
    sound_events_df["end_time"] = sound_events_df["geometry"].apply(lambda x: x["coordinates"][2])
except Exception:
    sound_events_df["begin_time"] = sound_events_df["geometry"].apply(lambda x: eval(x)["coordinates"][0])
    sound_events_df["end_time"] = sound_events_df["geometry"].apply(lambda x: eval(x)["coordinates"][2])

sound_events_df

,uuid,recording,geometry,features,begin_time,end_time
0,7ed175e8-0f04-41bb-9e17-ba011d43feba,44d8f0c5-9ffb-4707-b7d1-cf396906204c,"{'type': 'BoundingBox', 'coordinates': [3.2381...","{'Lower frequency bound': 184.1262014293534, '...",3.238100,3.371416
1,24e1ab15-4d4a-4174-8d6f-feb4e2d1ea0a,44d8f0c5-9ffb-4707-b7d1-cf396906204c,"{'type': 'BoundingBox', 'coordinates': [8.3617...","{'Lower frequency bound': 173.86736018240026, ...",8.361799,8.500190
2,5a691b5d-3c02-483b-a402-cee5a2638545,44d8f0c5-9ffb-4707-b7d1-cf396906204c,"{'type': 'BoundingBox', 'coordinates': [13.072...","{'Lower frequency bound': 265.5715889323974, '...",13.072388,13.222778
3,e912dcbb-cecf-4316-9e00-802181bbef70,44d8f0c5-9ffb-4707-b7d1-cf396906204c,"{'type': 'BoundingBox', 'coordinates': [17.468...","{'Lower frequency bound': 211.05301218324803, ...",17.468622,17.616369
4,10aa06ab-3a33-43fa-bc1d-65debced74b1,44d8f0c5-9ffb-4707-b7d1-cf396906204c,"{'type': 'BoundingBox', 'coordinates': [23.319...","{'Lower frequency bound': 132.53735620385123, ...",23.319983,23.445047
...,...,...,...,...,...,...
548,2c96ca56-2ec8-48aa-be50-f99f9b392350,b9f26380-6b03-42ae-ac71-4173cdaea550,"{'type': 'BoundingBox', 'coordinates': [21.747...","{'Lower frequency bound': 285.17503791221816, ...",21.747537,21.860636
549,e876cf07-e02a-4271-8d82-de1bf4094d41,b9f26380-6b03-42ae-ac71-4173cdaea550,"{'type': 'BoundingBox', 'coordinates': [24.136...","{'Lower frequency bound': 225.04858245407786, ...",24.136483,24.270781
550,a8efb780-e337-46dc-b318-cd8ca567d713,b9f26380-6b03-42ae-ac71-4173cdaea550,"{'type': 'BoundingBox', 'coordinates': [26.935...","{'Lower frequency bound': 122.57882722368004, ...",26.935901,27.061785
551,3a036a4f-4f1a-4c6d-b899-c8e80a613d1d,b9f26380-6b03-42ae-ac71-4173cdaea550,"{'type': 'BoundingBox', 'coordinates': [29.162...","{'Lower frequency bound': 219.86513069462762, ...",29.162379,29.272001


In [ ]:
# Convert to Raven selection tables
OUTPUT_DIR = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables"


selection_table_dir = OUTPUT_DIR
os.makedirs(selection_table_dir, exist_ok=True)

count_skipped = 0
for uuid, capture_id in uuid_to_capture_map.items():

    # Find sound events corresponding to this recording
    sound_events = sound_events_df[sound_events_df['recording'] == uuid]

    if len(sound_events) == 0:
        count_skipped += 1
        continue
    
    output_path = os.path.join(selection_table_dir, f"{capture_id}_selection_table.txt")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("Begin Time (s)\tEnd Time (s)\tAnnotation\n")

    for _, row in sound_events.iterrows():
        with open(output_path, "a", encoding="utf-8") as f:
            f.write(f"{row['begin_time']}\t{row['end_time']}\tevent\n")

print(f"Finished writing {len(uuid_to_capture_map) - count_skipped} selection tables to: {selection_table_dir}")

Finished writing 26 selection tables to: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables


## Option B: Raw Raven Export

Follow these steps if strongly labelled data has been exported directly off Raven.

In [2]:
# Define input directory containing Raven selection tables
INPUT_DIR = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables"


selection_table_dir = INPUT_DIR
template_suffix = ".Table.1.selections.txt"

for filename in os.listdir(selection_table_dir):
    if filename.endswith(template_suffix):
        
        full_path = os.path.join(selection_table_dir, filename)
        print(f"Processing: {filename}")

        try:
            # Load Raven selection table (tab-separated)
            df = pd.read_csv(full_path, sep="\t")

            # Keep only required columns
            df_clean = df[["Begin Time (s)", "End Time (s)"]].copy()

            # Remove duplicate rows (waveform/spectrogram duplicates)
            df_clean = df_clean.drop_duplicates()

            # Add Annotation column
            df_clean["Annotation"] = "event"

            # Save cleaned file
            output_name = filename.replace(template_suffix, "_selection_table.txt")
            output_path = os.path.join(selection_table_dir, output_name)

            df_clean.to_csv(output_path, sep="\t", index=False)

            print(f"Saved: {output_path}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

Processing: 286507.Table.1.selections.txt
Saved: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables/286507_selection_table.txt
Processing: 330650.Table.1.selections.txt
Saved: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables/330650_selection_table.txt
Processing: 357250.Table.1.selections.txt
Saved: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables/357250_selection_table.txt
Processing: 501141.Table.1.selections.txt
Saved: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables/501141_selection_table.txt
Processing: 259359.Table.1.selections.txt
Saved: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables/259359_selection_table.txt
Processing: 449431.Table.1.selections.txt
Saved: /home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables/449431_selection_table.txt
Proc

# Step 2 - Create Info Table
The info table is a CSV file with 3 columns:

* ``fn``: Unique filename associated with each audio file
* ``audio_fp``: Filepaths to audio files in train set
* ``selection_table_fp``: Filepath to Raven selection tables

For running model inference only, you don't need to have the `selection_table_fp` column set.

In [3]:
# Load captures table
captures_path = "/home/admin/julia-dev/data/frogid_captures/data_tables/FrogID_Captures_2025-03-09T21-01-13+0000.csv"
captures_df = pd.read_csv(captures_path)

/tmp/ipykernel_314368/3616435119.py:3: DtypeWarning: Columns (32,34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  captures_df = pd.read_csv(captures_path)


In [18]:
AUDIO_DIR = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/audio"

# If the audio files already exist in this directory, set this flag to True
AUDIO_EXISTS = True

captures_labelled = [x.split('_')[0] for x in os.listdir(selection_table_dir) if x.endswith('_selection_table.txt')]

# Otherwise, automatically download the audio files from the urls in the captures table
def download_audio(audio_dir, capture_id, audio_url):
    save_path = os.path.join(audio_dir, f'{capture_id}.aac')
    urlretrieve(audio_url, save_path)
    audio = AudioSegment.from_file(save_path)
    wav_path = os.path.join(audio_dir, f'{capture_id}.wav')
    audio.export(wav_path, format='wav')
    os.remove(save_path)

if not AUDIO_EXISTS:
    for capture_id in tqdm(captures_labelled, desc="Downloading audio"):
        row = captures_df[captures_df['id'] == int(capture_id)]
        audio_url = row['call_audio'].values[0]
        download_audio(AUDIO_DIR, capture_id, audio_url)

In [ ]:
CSV_OUT = 'datasets/frogid/Lim_peronii_info.csv'

# Set this flag to True if generating info table for model inference only
OMIT_SELECTION_TABLES = True

# Generate pandas dataframe
info_table = []
for capture_id in captures_labelled:

    if OMIT_SELECTION_TABLE:
        info_table.append({
            'fn': capture_id,
            'audio_fp': os.path.join(AUDIO_DIR, f'{capture_id}.wav')   
        })
    else:
        info_table.append({
            'fn': capture_id,
            'audio_fp': os.path.join(AUDIO_DIR, f'{capture_id}.wav'),
            'selection_table_fp': os.path.join(selection_table_dir, f'{capture_id}_selection_table.txt')
        })

info_df = pd.DataFrame.from_dict(info_table)

# Export as CSV
info_df.to_csv(CSV_OUT)
print(f'Saved info table to: {CSV_OUT}')

Saved info table to: datasets/frogid/Lim_peronii_info.csv


In [ ]:
info_df

,fn,audio_fp,selection_table_fp
0,407077,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
1,224604,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
2,200960,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
3,375076,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
4,224809,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
...,...,...,...
69,606199,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
70,104290,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
71,76260,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...
72,333253,/home/admin/julia-dev/data/frogid_captures/str...,/home/admin/julia-dev/data/frogid_captures/str...


# Step 3 - Split into Train/Test/Val

In [20]:
INPUT_CSV = 'datasets/frogid/Lim_peronii_info.csv'
OUTPUT_DIR = '/home/admin/julia-dev/voxaboxen/datasets/frogid/Lim_peronii_formatted'
RANDOM_SEED = 42

TRAIN_RATIO = 0.80
TEST_RATIO  = 0.10
VAL_RATIO   = 1.0 - (TRAIN_RATIO + TEST_RATIO)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load CSV and treat first column as index
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index.name = "index"

# Shuffle WITHOUT resetting index
df_shuf = df.sample(frac=1.0, random_state=RANDOM_SEED)

# Split
n = len(df_shuf)
n_train = int(n * TRAIN_RATIO)
n_test  = int(n * TEST_RATIO)

train_df = df_shuf.iloc[:n_train]
test_df  = df_shuf.iloc[n_train:n_train + n_test]
val_df   = df_shuf.iloc[n_train + n_test:]

# Write CSVs — index preserved, no extra column
train_df.to_csv(os.path.join(OUTPUT_DIR, "train_info.csv"),
                index=True, index_label=None)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_info.csv"),
               index=True, index_label=None)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_info.csv"),
              index=True, index_label=None)

print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")


Split sizes: train=79, val=11, test=9


# [Optional] Step 4 - Download Additional Audio for Inference

This optional step is for creating a test info table from a separate audio dataset, which will be used for running inference (NOT training). 

In [ ]:
# Define data source and destination
CSV_PATH = "/home/admin/julia-dev/data/frogid_captures/data_tables/fraser_cuecounts_all_data.csv"
AUDIO_DIR = "/home/admin/julia-dev/data/frogid_captures/individual_count"

df = pd.read_csv(CSV_PATH)
count = 0

for _, row in df.iterrows():
    
    capture_id = row['id']
    species = row['species']

    audio_subdir = os.path.join(AUDIO_DIR, '_'.join(species.split()).lower())
    audio_path = os.path.join(audio_subdir, f'capture_{capture_id}.wav')

    if os.path.exists(audio_path):
        continue
    else:
        try:
            url = captures_df[captures_df['id'] == capture_id]['call_audio'].values[0]
        except IndexError:
            print(f'{capture_id} not found in captures table --> SKIPPING')
        else:
            download_audio(audio_subdir, capture_id, url)

/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_caerulea_multispecies_test


In [ ]:
# Group by species for separate test directories
species_dfs = {
    species: group.copy()
    for species, group in df.groupby("species")
}

{'Limnodynastes peronii':       Unnamed: 0      id                species        lat         lng  \
 1029           1    8458  Limnodynastes peronii -34.438900  150.469000   
 1030           2    8988  Limnodynastes peronii -33.804395  151.173691   
 1031           3    9346  Limnodynastes peronii -33.223300  151.504000   
 1032           4    9581  Limnodynastes peronii -33.725997  151.152149   
 1033           5   10032  Limnodynastes peronii -33.083200  151.434000   
 ...          ...     ...                    ...        ...         ...   
 1480         452  795854  Limnodynastes peronii -36.814200  149.794000   
 1481         453  797510  Limnodynastes peronii -33.805396  151.191289   
 1482         454  799406  Limnodynastes peronii -33.721400  150.700000   
 1483         455  801041  Limnodynastes peronii -33.803944  151.189016   
 1484         456  803310  Limnodynastes peronii -32.987000  151.586000   
 
           year  eventhour  earcount  callcount   duration       temp  \


In [ ]:
# Create test info table for each species
DATASET_DIR = "/home/admin/julia-dev/voxaboxen/datasets/frogid"

def make_audio_fp(species, id_):
    folder = '_'.join(str(species).split()).lower()
    return os.path.join(AUDIO_DIR, folder, f"capture_{id_}.wav")

for species, df in species_dfs.items():
    new_df = pd.DataFrame({
        "fn": df["id"]
    })

    new_df["audio_fp"] = [
        make_audio_fp(species, id_)
        for species, id_ in zip(df["species"], df["id"])
    ]

    # remove bad paths / malformed species / missing files
    new_df = new_df[
        new_df["audio_fp"].apply(os.path.exists)
    ].copy()

    # optional: convert paths to strings
    new_df["audio_fp"] = new_df["audio_fp"].astype(str)
    new_df = new_df.reset_index(drop=True)

    parts = species.split()
    save_dir = os.path.join(DATASET_DIR, f'{'_'.join((parts[0][:3], parts[1]))}_multispecies_test')
    save_path = os.path.join(save_dir, 'test_info.csv')
    new_df.to_csv(save_path)